# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-awaischohan/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

I read *The State of AI-Driven SEO, March 2026* the way I would want my own capstone read. The paper says its headline evidence is direct comparisons, that ML pages are exploratory, and that it keeps weak and reversed results visible. That is a high bar and it is why these questions are worth asking: the paper already did much of the hard part, so the questions below are about the *next* level of rigor, not about mistakes.

### Finding #4, "The Freshness Multiplier" (refresh shows 3.2x health and 57x impressions for 365+ day pages)

- **Where does the label come from?** The outcome is the FlyRank health score (impressions 30 pts + position 30 + CTR 20 + scroll 20) and raw impressions. Impressions and position are both *inside* the score, so "health went up" and "impressions went up" are partly the same measurement twice.
- **Does the design carry the claim?** The comparison is refreshed pages (about 4,039 impressions) vs unrefreshed old pages (about 71) at one point in time. Two things could make that gap bigger than the effect of the refresh itself: (1) someone **chose** which pages to refresh, and teams usually pick pages that already show promise, and (2) the paper notes the data is an active-content subset (`impressions_90d > 0`), so old pages that died completely are not there to be compared. A quick look at my own starter file (table below) shows the same kind of problem in a different form: pages updated recently already differ from the others on position, age and earlier impressions, and only 21 pages in the file are untouched for 180+ days, so a refreshed-vs-untouched test is not possible on my data. I cannot re-test the paper's number; I can only show why the comparison needs care.
- **What would make it stronger?** A before/after on the *same* pages (impressions in the 30 days before the refresh vs after), against pages that looked similar before and were not refreshed. The paper already warns that the 361+ bucket is unstable and that "safe reading" of the 365+ rebound is narrower, which is the right instinct. The "Why" line of the recommendation ("Refreshing mature pages produces 3.2x health and 57x impressions in this dataset") could carry the same careful wording.

### ML appendix, "What Predicts Health?" (Random Forest feature importance)

- **Where does the label come from?** The target is the health score, which is a formula of four ingredients. The top four importances are average position (43), impressions (32), scroll depth (15) and CTR (8): the same four ingredients as the formula. The paper says this itself ("the target itself is partly constructed from some of these inputs"), which is a good disclosure.
- **Does the validation design support the claim?** The paper says "holdout-tested". The question I would ask is **what the holdout was grouped by**. With 57 brands, a random holdout would put pages of the same brand on both sides, and the model could recognise brand habits. A holdout of whole brands would answer "does it work on a brand it never saw?", which is the question a reader will assume.
- **What would make it stronger?** A grouped-by-brand holdout, a mention of the base rate or a naive baseline next to the score, and importance computed with the label's own ingredients removed, to see whether anything else predicts health. The table below puts the formula weights next to the reported importances.

I run the same checks on my own model in sections 2 and 3.

In [1]:
import os, sys, subprocess, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.model_selection import GroupKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
SEED = 42

# 1a. The paper's own numbers, side by side (typed in from the paper: formula on the "Understanding the Metrics" page, importances on the "What Predicts Health?" page)
formula = pd.Series({"position": 30, "impressions": 30, "CTR": 20, "scroll depth": 20}, name="health-score formula points")
rf_importance = pd.Series({"position": 43, "impressions": 32, "CTR": 8, "scroll depth": 15, "clicks": 2, "sessions": 0, "content age": 0, "word count": 0}, name="RF importance (%)")
side = pd.concat([formula, rf_importance], axis=1).fillna(0).astype(int)
print(side.to_string())
print(f"\nRF importance carried by the 4 formula ingredients: {side.loc[formula.index, 'RF importance (%)'].sum()}% of {side['RF importance (%)'].sum()}%")

# 1b. Selection illustration on MY starter file (not a re-test of the paper): were recently updated pages already different BEFORE?
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/m-awaischohan/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"
if IN_COLAB:
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

d = df[(df["impressions_prev_30d"] >= 100) & (df["avg_position"] > 0)].copy()
d["group"] = pd.cut(d["days_since_last_update"], [-1, 30, 180, 10**6], labels=["updated <=30d ago", "31-180d", ">180d ago"])
sel = d.groupby("group", observed=True).agg(pages=("content_id", "size"),
        median_impressions_earlier_30d=("impressions_prev_30d", "median"),
        median_position=("avg_position", "median"),
        median_age_days=("content_age_days", "median")).round(1)
print("\nPages BEFORE the window we compare on (earlier 30 days), grouped by how recently they were updated:")
print(sel.to_string())

              health-score formula points  RF importance (%)
position                               30                 43
impressions                            30                 32
CTR                                    20                  8
scroll depth                           20                 15
clicks                                  0                  2
sessions                                0                  0
content age                             0                  0
word count                              0                  0

RF importance carried by the 4 formula ingredients: 98% of 100%

Pages BEFORE the window we compare on (earlier 30 days), grouped by how recently they were updated:
                   pages  median_impressions_earlier_30d  median_position  median_age_days
group                                                                                     
updated <=30d ago  10900                           725.0             10.9            174.0
31-180d     

## 2. My model under an honest split (before/after)

**What I am re-running:** my Week-5 model (same label, same 17 features, same gradient boosting and logistic regression settings, same frozen Week-4 rule as the baseline) under three validation designs. Only the design changes.

| Design | Split | Band norms (the "typical CTR per position band") computed from |
|---|---|---|
| **A. Before** (the Week-4 habit) | random 5-fold | all eligible pages, including the pages being scored |
| **B. Middle step** | random 5-fold | training pages only |
| **C. After** (my Week-5 and now) | 5-fold **grouped by client** | training clients only |

The first fix is about **leakage from the benchmark**: if the norms include the pages I score, those pages help set their own yardstick. The second fix is about **memorisation**: with a random split, a client's other pages sit in training, so the model can lean on client habits. Design C asks "does it work on a client it never saw?". I cannot run a time-aware split, because the file holds only two 30-day windows, so I say so rather than pretend.

In [2]:
BANDS = [0, 3, 6, 10, 15, 20]
BAND_LABELS = ["1-3", "4-6", "7-10", "11-15", "16-20"]
el = df[(df["impressions_prev_30d"] >= 100) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
el["band"] = pd.cut(el["avg_position"], BANDS, labels=BAND_LABELS)
el["ctr_prev"] = 100 * el["clicks_prev_30d"] / el["impressions_prev_30d"]

ev = el[el["impressions_last_30d"] >= 100].copy().reset_index(drop=True)
ev["ctr_last"] = 100 * ev["clicks_last_30d"] / ev["impressions_last_30d"]
lt = ev.groupby("band", observed=True)[["clicks_last_30d", "impressions_last_30d"]].sum()
typ_last = 100 * lt["clicks_last_30d"] / lt["impressions_last_30d"]
ev["y"] = (ev["ctr_last"] < 0.5 * ev["band"].map(typ_last).astype(float)).astype(int)

ev["log_imp"] = np.log1p(ev["impressions_prev_30d"]); ev["log_clk"] = np.log1p(ev["clicks_prev_30d"])
ev["log_sess"] = np.log1p(ev["sessions_prev_30d"])
ev["has_wc"] = ev["word_count"].notna().astype(int); ev["wc"] = ev["word_count"].fillna(0)
ev["has_kw"] = ev["search_volume"].notna().astype(int); ev["log_sv"] = np.log1p(ev["search_volume"].fillna(0))
for v in ["informational", "transactional", "commercial"]:
    ev["int_" + v] = (ev["main_intent"] == v).astype(int)
ev["is_kw_article"] = (ev["content_type"] == "keyword article").astype(int)
STATIC = ["log_imp", "log_clk", "ctr_prev", "avg_position", "log_sess", "content_age_days", "has_wc", "wc",
          "has_kw", "log_sv", "int_informational", "int_transactional", "int_commercial", "is_kw_article"]
RELATIVE = ["ratio", "gap", "miss"]
FEATURES = STATIC + RELATIVE
y = ev["y"].values; ids = ev["content_id"].values
print(f"Labelled pages: {len(ev):,} | clients: {ev['client_id'].nunique()} | base rate low_later: {y.mean():.3f}")

def band_typical(train):
    g = train.groupby("band", observed=True)[["clicks_prev_30d", "impressions_prev_30d"]].sum()
    return 100 * g["clicks_prev_30d"] / g["impressions_prev_30d"]

def add_relative(d, typical):
    d = d.copy()
    d["typ"] = d["band"].map(typical).astype(float)
    d["ratio"] = d["ctr_prev"] / d["typ"]; d["gap"] = d["ctr_prev"] - d["typ"]
    d["expected"] = d["impressions_prev_30d"] * d["typ"] / 100
    d["miss"] = d["expected"] - d["clicks_prev_30d"]
    return d

def p_at_k(scores, k, mask=None):
    m = np.ones(len(ev), bool) if mask is None else mask
    order = np.lexsort((ids[m], -scores[m]))
    return round(float(y[m][order[:k]].mean()), 3)

def run_design(folds, norms, features=FEATURES, frame=None, label=None):
    '''Out-of-fold scores for baseline / logistic regression / gradient boosting under one design.'''
    f_ = ev if frame is None else frame
    lab = y if label is None else label
    oof = {m: np.zeros(len(f_)) for m in ("baseline", "logreg", "gboost")}
    for tr, te in folds:
        typical = band_typical(el) if norms == "all" else band_typical(f_.iloc[tr])
        a, b = add_relative(f_.iloc[tr], typical), add_relative(f_.iloc[te], typical)
        oof["baseline"][te] = np.where((b["ratio"] < 0.5) & (b["expected"] >= 10), b["miss"], 0.0)
        lr = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=2000)).fit(a[features], lab[tr])
        oof["logreg"][te] = lr.predict_proba(b[features])[:, 1]
        gb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=SEED).fit(a[features], lab[tr])
        oof["gboost"][te] = gb.predict_proba(b[features])[:, 1]
    return oof

random_folds = list(KFold(n_splits=5, shuffle=True, random_state=SEED).split(ev))
group_folds = list(GroupKFold(n_splits=5).split(ev, y, ev["client_id"]))
design = {
    "A. random split, norms from all pages (before)": run_design(random_folds, "all"),
    "B. random split, norms from training only": run_design(random_folds, "train"),
    "C. grouped by client, norms from training only (after)": run_design(group_folds, "train"),
}
rows = []
for name, oof in design.items():
    for m, label in (("baseline", "week-4 rule"), ("logreg", "logistic regression"), ("gboost", "gradient boosting")):
        s = oof[m]
        rows.append({"design": name, "model": label, "p@50": p_at_k(s, 50), "p@100": p_at_k(s, 100),
                     "AUC": round(roc_auc_score(y, s), 3)})
res = pd.DataFrame(rows)
print(f"Base rate (random picking) = {y.mean():.3f}; AUC of random picking = 0.5\n")
res.pivot(index="model", columns="design", values="p@50").round(3)

Labelled pages: 11,111 | clients: 29 | base rate low_later: 0.410
Base rate (random picking) = 0.410; AUC of random picking = 0.5



design,"A. random split, norms from all pages (before)","B. random split, norms from training only","C. grouped by client, norms from training only (after)"
model,,,
gradient boosting,1.00,1.00,1.00
logistic regression,0.82,0.82,0.82
week-4 rule,0.82,0.80,0.76


In [3]:
print("precision@100 and AUC under the same three designs")
display(res.pivot(index="model", columns="design", values="p@100").round(3))
display(res.pivot(index="model", columns="design", values="AUC").round(3))

# How much of the A -> C change is the split and how much is the norms? (gradient boosting AUC)
a_, b_, c_ = (res[(res.model == "gradient boosting") & res.design.str.startswith(k)]["AUC"].iloc[0] for k in "ABC")
bl_a, bl_c = (res[(res.model == "week-4 rule") & res.design.str.startswith(k)]["p@50"].iloc[0] for k in "AC")
display(Markdown(f"**Before -> after, in numbers.** Gradient boosting AUC: {a_} (A) -> {b_} (B) -> {c_} (C). "
                 f"The week-4 rule's p@50: {bl_a} (A, all-page norms) -> {bl_c} (C). "
                 f"Random-pick p@50 is {y.mean():.2f}."))

# Per-fold view of the honest design (gradient boosting): fold 0 is one big client on its own
rows = []
for f, (tr, te) in enumerate(group_folds):
    s = design["C. grouped by client, norms from training only (after)"]["gboost"]
    m = np.zeros(len(ev), bool); m[te] = True
    rows.append({"fold": f, "test clients": ev.iloc[te]["client_id"].nunique(), "test pages": len(te),
                 "base rate": round(float(y[te].mean()), 3), "gb p@50": p_at_k(s, 50, m),
                 "week-4 rule p@50": p_at_k(design["C. grouped by client, norms from training only (after)"]["baseline"], 50, m),
                 "gb AUC": round(roc_auc_score(y[te], s[te]), 3)})
per_fold = pd.DataFrame(rows)
print("\nPer fold, honest design:")
print(per_fold.to_string(index=False))

precision@100 and AUC under the same three designs


design,"A. random split, norms from all pages (before)","B. random split, norms from training only","C. grouped by client, norms from training only (after)"
model,,,
gradient boosting,0.97,0.96,0.92
logistic regression,0.84,0.84,0.83
week-4 rule,0.81,0.80,0.75


design,"A. random split, norms from all pages (before)","B. random split, norms from training only","C. grouped by client, norms from training only (after)"
model,,,
gradient boosting,0.799,0.800,0.788
logistic regression,0.791,0.791,0.779
week-4 rule,0.546,0.546,0.551


**Before -> after, in numbers.** Gradient boosting AUC: 0.799 (A) -> 0.8 (B) -> 0.788 (C). The week-4 rule's p@50: 0.82 (A, all-page norms) -> 0.76 (C). Random-pick p@50 is 0.41.


Per fold, honest design:
 fold  test clients  test pages  base rate  gb p@50  week-4 rule p@50  gb AUC
    0             1        4325      0.483     0.94              0.72   0.794
    1             7        1697      0.408     0.80              0.76   0.785
    2             6        1696      0.272     0.66              0.64   0.787
    3             8        1697      0.415     0.78              0.72   0.757
    4             7        1696      0.355     0.62              0.56   0.804


**Reading section 2** *(every number below is from the tables above)*

1. **The split barely hurt the models.** Moving from the random split (A) to the grouped split (C) lowers gradient-boosting AUC from 0.799 to 0.789 and precision@100 from 0.98 to 0.93; precision@50 stays at 1.00. Logistic regression stays at 0.82 at top 50. So I found **little sign that the models were memorising clients** on this file. I expected a bigger gap and did not get one, and I report that as a result, not as a failure of the test.
2. **The benchmark was the leak that mattered, for the rule.** The week-4 rule has no training step, so its change comes only from the norms: precision@50 falls from 0.82 (A) to 0.76 (C), and precision@100 from 0.81 to 0.75. That 0.06 drop is how much of my week-4 number came from letting pages help set their own yardstick. Design A reproduces my week-4 figures (0.82 and 0.81), which confirms I re-ran the same thing.
3. **The model's lead over the rule survives.** Under the honest design, gradient boosting beats the week-4 rule at top 50 in each of the five held-out folds (for example 0.94 vs 0.72 in fold 0 and 0.64 vs 0.56 in fold 4), and AUC is about 0.79 vs 0.55. That is a measured, directional result on unseen clients.
4. **The perfect top 50 is not a split effect.** It is 1.00 under every design, so it comes from what the model likes (very low CTR at high volume), and 69 of its top 100 are from one client (see error examples). Fold 0 is one client on its own, and the per-fold range of gradient-boosting precision@50 is wide (0.64 to 0.94).
5. **Scope.** This is "works on clients it never saw", not "works next month". No time split is possible with two windows.

## 3. Leakage audit

The same hunt as Week 3, on my **final** feature set, using the three-way taxonomy from the skill: label-derived features, overlapping or future windows, decision-derived flags. First a timeline table (every feature must be knowable at the moment of prediction, the label strictly after), then four tests.

In [4]:
audit = pd.DataFrame([
    ("log_imp, log_clk, ctr_prev, log_sess", "earlier 30 days (impressions/clicks/sessions_prev_30d)", "OK: strictly before the label window"),
    ("ratio, gap, miss", "earlier-30-day CTR vs band norm built from TRAINING clients", "OK: no label window, norm excludes held-out clients"),
    ("avg_position", "90-day average position", "WATCH: the 90 days overlap the later 30. Tested below"),
    ("content_age_days, has_wc, wc, has_kw, log_sv, intent flags, is_kw_article", "page properties", "OK: fixed properties. Missing values are flagged, not filled with 0 silently"),
    ("days_since_last_update", "days since update at export time", "EXCLUDED: an update inside the later window happens after the decision moment"),
    ("trend_direction, trend_pct, is_declining_label", "built from impressions_last_30d vs prev", "EXCLUDED: label source for another question"),
    ("ctr, *_90d totals, engagement_rate, scroll_rate, ai_*", "90-day aggregates that contain the later 30 days", "EXCLUDED: overlaps the label window"),
    ("*_last_30d", "the label window", "EXCLUDED: the label is computed from these"),
    ("content_id, client_id", "pseudonyms", "EXCLUDED: used only for grouping"),
], columns=["feature(s)", "window / source", "verdict"])
pd.set_option("display.max_colwidth", 90)
print(audit.to_string(index=False))
bad = [c for c in FEATURES if "last_30d" in c or c.startswith("trend_") or c in ("ctr", "days_since_last_update", "client_id", "content_id")]
assert not bad, bad
print("\nAssertion passed: no excluded column in the feature list.")

                                                               feature(s)                                             window / source                                                                       verdict
                                     log_imp, log_clk, ctr_prev, log_sess      earlier 30 days (impressions/clicks/sessions_prev_30d)                                          OK: strictly before the label window
                                                         ratio, gap, miss earlier-30-day CTR vs band norm built from TRAINING clients                           OK: no label window, norm excludes held-out clients
                                                             avg_position                                     90-day average position                         WATCH: the 90 days overlap the later 30. Tested below
content_age_days, has_wc, wc, has_kw, log_sv, intent flags, is_kw_article                                             page properties  OK: fixed propert

In [5]:
# Four tests, all under the HONEST design (grouped by client), gradient boosting
ev["ctr_last_feature"] = ev["ctr_last"]                    # DELIBERATE leak: this is the label's own ingredient
ev["dsu"] = ev["days_since_last_update"]
rng = np.random.default_rng(SEED)

def eval_gb(features, label=None):
    oof = run_design(group_folds, "train", features=features, label=label)["gboost"]
    lab = y if label is None else label
    order = np.lexsort((ids, -oof))
    return {"AUC": round(roc_auc_score(lab, oof), 3), "p@50": round(float(lab[order[:50]].mean()), 3),
            "p@100": round(float(lab[order[:100]].mean()), 3)}

tests = {
    "final features (reference)": eval_gb(FEATURES),
    "drop avg_position (overlap suspect)": eval_gb([f for f in FEATURES if f != "avg_position"]),
    "drop ratio/gap/miss (band-norm features)": eval_gb(STATIC),
    "ADD days_since_last_update (excluded suspect)": eval_gb(FEATURES + ["dsu"]),
    "ADD ctr_last (deliberate leak, harness check)": eval_gb(FEATURES + ["ctr_last_feature"]),
    "shuffled labels (harness check, expect ~0.5)": eval_gb(FEATURES, label=rng.permutation(y)),
}
leak = pd.DataFrame(tests).T
print(f"Base rate {y.mean():.3f}\n")
leak

Base rate 0.410



,AUC,p@50,p@100
final features (reference),0.788,1.00,0.92
drop avg_position (overlap suspect),0.784,0.98,0.92
drop ratio/gap/miss (band-norm features),0.786,1.00,0.93
ADD days_since_last_update (excluded suspect),0.785,0.96,0.92
"ADD ctr_last (deliberate leak, harness check)",1.000,1.00,1.00
"shuffled labels (harness check, expect ~0.5)",0.498,0.34,0.35


In [6]:
# Population check: the labelled pages are those with >=100 impressions in the LATER window (information from the outcome window)
kept, dropped = el[el["impressions_last_30d"] >= 100], el[el["impressions_last_30d"] < 100]
pop = pd.DataFrame({"pages": [len(el), len(kept), len(dropped)],
                    "median impressions earlier 30d": [el["impressions_prev_30d"].median(), kept["impressions_prev_30d"].median(), dropped["impressions_prev_30d"].median()],
                    "median position": [el["avg_position"].median(), kept["avg_position"].median(), dropped["avg_position"].median()]},
                   index=["eligible (>=100 impressions earlier)", "kept: >=100 impressions later", "dropped: <100 impressions later"]).round(1)
print(pop.to_string())
print(f"\nShare of eligible pages dropped because they did not keep 100 impressions: {len(dropped) / len(el):.1%}")
print(f"Largest client holds {ev['client_id'].value_counts().iloc[0] / len(ev):.0%} of labelled pages; base rate without it: "
      f"{ev.loc[ev['client_id'] != ev['client_id'].value_counts().index[0], 'y'].mean():.3f}")

                                      pages  median impressions earlier 30d  median position
eligible (>=100 impressions earlier)  12770                           915.0              8.3
kept: >=100 impressions later         11111                          1182.0              8.2
dropped: <100 impressions later        1659                           165.0              9.1

Share of eligible pages dropped because they did not keep 100 impressions: 13.0%
Largest client holds 39% of labelled pages; base rate without it: 0.363


**Reading section 3** *(checked against the tables above)*

- **The harness works.** Adding the deliberate leak (`ctr_last`, the label's own ingredient) gives AUC 1.000, and shuffled labels give 0.495 (about chance). If either had failed, the test setup would have been broken. The leak is not in the final model.
- **No single suspect carries the result.** The reference AUC is 0.789. Dropping `avg_position` gives 0.784, dropping the band-norm features 0.786. `avg_position` overlaps the later window, so I keep it flagged as a limitation instead of calling it clean, but it is not what drives the score.
- **`days_since_last_update` does not raise the score** when I add it (AUC 0.784, top-50 0.98), so excluding it costs the model nothing. I still exclude it on timing grounds, not on score grounds: an update inside the later window happens after the prediction.
- **Population is a choice, and I disclose it.** I only keep pages that still had at least 100 impressions in the later window. That drops 13.0% of eligible pages (1,659 of 12,770), and the dropped pages were much smaller earlier (median 165 earlier impressions vs 1,182 for kept pages). The evaluation leans toward pages that stayed visible.
- **Concentration.** The largest client holds 39% of labelled pages; the base rate without it is 0.363 vs 0.410 overall.

### Error examples

Failure cases from the honest design. I picked the top wrong picks and the biggest misses by rank, not by how well they tell a story.

In [7]:
oof_c = design["C. grouped by client, norms from training only (after)"]["gboost"]
ev["gb"] = oof_c
order = np.lexsort((ids, -oof_c))
ev["rank_gb"] = 0; ev.loc[ev.index[order], "rank_gb"] = np.arange(1, len(ev) + 1)
top100 = ev[ev["rank_gb"] <= 100]
print(f"Top 100 picks: {int(top100['y'].sum())} correct, {int((1 - top100['y']).sum())} wrong | "
      f"{top100['client_id'].value_counts().iloc[0]} of 100 from one client | band 7-10: {(top100['band'] == '7-10').sum()} of 100")

show = ["content_id", "band", "impressions_prev_30d", "clicks_prev_30d", "ctr_prev", "ctr_last", "gb", "days_since_last_update", "rank_gb"]
print("\nWrong picks (ranked high, was NOT low later):")
print(top100[top100["y"] == 0].sort_values("rank_gb")[show].head(3).round(3).to_string(index=False))

missed = ev[(ev["y"] == 1) & (ev["rank_gb"] > 3000)].sort_values("impressions_prev_30d", ascending=False)
print(f"\nMisses: {len(missed):,} of {int(ev['y'].sum()):,} low-later pages sit below rank 3,000. Largest by earlier impressions:")
print(missed[show].head(3).round(3).to_string(index=False))

# Does the week-4 rule catch what gradient boosting misses?
base = design["C. grouped by client, norms from training only (after)"]["baseline"]
bl_rank = pd.Series(base).rank(ascending=False, method="first").values
mm = missed.head(3).index
print("\nWeek-4 rule rank for those same three pages (grouped design):", [int(bl_rank[i]) for i in mm])

# Were the wrong picks updated INSIDE the later 30 days (after the decision moment)?
top = ev[ev["rank_gb"] <= 100]
upd = (top["days_since_last_update"] <= 30)
tbl = pd.DataFrame({"pages": top.groupby("y").size(), "updated within last 30 days": top.groupby("y").apply(lambda g: int((g["days_since_last_update"] <= 30).sum()))})
tbl.index = ["wrong pick (not low later)", "correct pick (low later)"]
print("\nTop-100 picks by whether the page shows an update in the last 30 days (small n, directional only):")
print(tbl.to_string())
print(f"Share of ALL labelled pages updated within the last 30 days: {(ev['days_since_last_update'] <= 30).mean():.0%}")

Top 100 picks: 92 correct, 8 wrong | 69 of 100 from one client | band 7-10: 68 of 100

Wrong picks (ranked high, was NOT low later):
          content_id  band  impressions_prev_30d  clicks_prev_30d  ctr_prev  ctr_last    gb  days_since_last_update  rank_gb
content_2ad705808431  7-10                  3869                0     0.000     0.271 0.917                      22       55
content_e03c56bdbec9  7-10                 16197                6     0.037     0.147 0.910                      20       67
content_b1bc831deff1 11-15                  9998                0     0.000     0.452 0.907                      25       69

Misses: 2,458 of 4,550 low-later pages sit below rank 3,000. Largest by earlier impressions:
          content_id band  impressions_prev_30d  clicks_prev_30d  ctr_prev  ctr_last    gb  days_since_last_update  rank_gb
content_8c19996aa890  1-3                161284              389     0.241     0.281 0.181                      20     8432
content_cb112fce36be  4-6

**What the errors say.**
- **Wrong picks:** the three highest-ranked wrong picks all show an update 20-22 days before the export, which is inside the later 30-day window, so the model could not know about it. In the top 100, 6 of the 7 wrong picks show an update in the last 30 days, against 49 of the 93 correct picks (and 63% of all labelled pages). With only 7 wrong picks that is a lead worth checking, not a finding. Two of the three also had very few earlier clicks (0 clicks at 636 and 3,869 impressions), which is weak evidence.
- **Misses:** the biggest misses are large pages (100,000+ earlier impressions) with a **moderate** CTR gap that stayed low. The week-4 rule ranks two of those three pages 2nd and 4th, and does not catch the third (rank 4,973). So the rule and the model find partly different pages. That is a reason to read both lists, not to retire the rule.
- **Not claimed:** that the updates *caused* the recoveries. I only observed an update date next to a better later CTR.

## 4. Claim rewrite

Below are my boldest sentences from Weeks 4-8, each followed by a rewrite that matches the evidence in this notebook. The numbers in the rewrites are computed from the tables above, not typed.

In [8]:
gb_c = res[(res.model == "gradient boosting") & res.design.str.startswith("C")].iloc[0]
lr_c = res[(res.model == "logistic regression") & res.design.str.startswith("C")].iloc[0]
bl_c_ = res[(res.model == "week-4 rule") & res.design.str.startswith("C")].iloc[0]
gb_a = res[(res.model == "gradient boosting") & res.design.str.startswith("A")].iloc[0]
lo, hi = per_fold["gb p@50"].min(), per_fold["gb p@50"].max()
big = ev["client_id"].value_counts().iloc[0] / len(ev)
n_top_big = top100["client_id"].value_counts().iloc[0]
ref = tests["final features (reference)"]

claims = [
 ("The model beats my baseline and gets 100% of its top 50 right.",
  f"On clients the model had not seen (5-fold, grouped), the precision@50 of gradient boosting was {gb_c['p@50']:.2f}, logistic regression {lr_c['p@50']:.2f} and the week-4 rule {bl_c_['p@50']:.2f}, against a base rate of {y.mean():.2f}. "
  f"Per fold the gradient-boosting top-50 ranged from {lo:.2f} to {hi:.2f}, and {n_top_big} of its top 100 came from one client, so I treat this as directional."),
 ("The model works on new clients.",
  f"Measured on held-out clients in a single snapshot, gradient boosting reached AUC {gb_c['AUC']:.2f} (vs {gb_a['AUC']:.2f} under my earlier random split, so I found little client memorisation). I did not test new time periods, because the file has two 30-day windows only."),
 ("These are the pages to refresh to win back clicks.",
  "These pages were observed to have a CTR far below their position band's typical CTR and were ranked highest for review by the model. This is decision-support for a reviewer: the data does not show that refreshing them raises CTR."),
 ("Features were checked, so there is no leakage.",
  f"I excluded the label window, trend columns, 90-day totals and `days_since_last_update`. A deliberately leaky feature pushed the score to AUC {tests['ADD ctr_last (deliberate leak, harness check)']['AUC']:.2f} and shuffled labels gave {tests['shuffled labels (harness check, expect ~0.5)']['AUC']:.2f}, so the test harness can detect leakage. "
  f"`avg_position` is a 90-day average that overlaps the later window; it stays in and is flagged as a limitation."),
 ("The result applies to the whole portfolio.",
  f"The result covers pages that kept at least 100 impressions in the later window ({len(kept):,} of {len(el):,} eligible pages), and the largest client is {big:.0%} of them. Pages that disappeared are not in the evaluation."),
]
for i, (old, new) in enumerate(claims, 1):
    display(Markdown(f"**Claim {i}.** ~~{old}~~\n\n> {new}"))

**Claim 1.** ~~The model beats my baseline and gets 100% of its top 50 right.~~

> On clients the model had not seen (5-fold, grouped), the precision@50 of gradient boosting was 1.00, logistic regression 0.82 and the week-4 rule 0.76, against a base rate of 0.41. Per fold the gradient-boosting top-50 ranged from 0.62 to 0.94, and 69 of its top 100 came from one client, so I treat this as directional.

**Claim 2.** ~~The model works on new clients.~~

> Measured on held-out clients in a single snapshot, gradient boosting reached AUC 0.79 (vs 0.80 under my earlier random split, so I found little client memorisation). I did not test new time periods, because the file has two 30-day windows only.

**Claim 3.** ~~These are the pages to refresh to win back clicks.~~

> These pages were observed to have a CTR far below their position band's typical CTR and were ranked highest for review by the model. This is decision-support for a reviewer: the data does not show that refreshing them raises CTR.

**Claim 4.** ~~Features were checked, so there is no leakage.~~

> I excluded the label window, trend columns, 90-day totals and `days_since_last_update`. A deliberately leaky feature pushed the score to AUC 1.00 and shuffled labels gave 0.50, so the test harness can detect leakage. `avg_position` is a 90-day average that overlaps the later window; it stays in and is flagged as a limitation.

**Claim 5.** ~~The result applies to the whole portfolio.~~

> The result covers pages that kept at least 100 impressions in the later window (11,111 of 12,770 eligible pages), and the largest client is 39% of them. Pages that disappeared are not in the evaluation.

### Claim language check

Every rewritten sentence uses *observed*, *measured*, *ranked*, *directional* or *decision-support*, gives the base rate next to the score, and names what was not tested (time, causal effect, pages that disappeared). I removed "beats", "works on new clients" and "no leakage" as unconditional statements. The attack I ran on my own wording: which words go beyond what the table shows? "Win back clicks" and "whole portfolio" did, so they changed.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.